<a href="https://colab.research.google.com/github/judithwang021/judith-6070/blob/main/WeeklyModules/Week05/Assignment5_improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5: Improved FAANG Stock Data Notebook

This version keeps the original GitHub connection workflow and improves the repeated data-reading, combining, cleaning, and saving steps.

In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import statsmodels.formula.api as sm
import statsmodels.api as se
import seaborn as sns
import datetime as dt
from sklearn.linear_model import LinearRegression
from scipy import stats
from io import StringIO
import itertools
from pathlib import Path

pd.set_eng_float_format(accuracy=3, use_eng_prefix=True)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

from google.colab import userdata
import os

github_token = userdata.get('github_token')
owner = 'judithwang021' # Replace with the GitHub repository owner
repository = 'judith-6070' # Replace with the GitHub repository name

clone_url = f'https://{github_token}@github.com/{owner}/{repository}.git'

# Clone the repository
import subprocess
subprocess.run(['git', 'clone', clone_url])

# Navigate into the cloned repository directory (optional, but often useful)
os.chdir(repository)
print(f"Changed directoGit_HUBry to: {os.getcwd()}")


Changed directoGit_HUBry to: /content/judith-6070


## Generative AI recommendations

- Store ticker symbols and file paths in one dictionary instead of repeating five `read_csv()` statements.
- Use a loop to read each file and add its ticker column consistently.
- Concatenate the data once with `ignore_index=True`.
- Clean and sort the combined data before saving it.
- Avoid saving a CSV and immediately reading the identical CSV back into Python.
- Use clear variable names and show useful results at each stage.

In [2]:
# Store the GitHub data-file paths
stock_files = {
    'AAPL': Path('/content/judith-6070/data/aapl.csv'),
    'AMZN': Path('/content/judith-6070/data/amzn.csv'),
    'NFLX': Path('/content/judith-6070/data/nflx.csv'),
    'FB': Path('/content/judith-6070/data/fb.csv'),
    'GOOG': Path('/content/judith-6070/data/goog.csv')
}

# Read each stock file and add its ticker
stock_dataframes = []
for ticker, file_path in stock_files.items():
    stock_df = pd.read_csv(file_path)
    stock_df['ticker'] = ticker
    stock_dataframes.append(stock_df)

# Combine all five stocks into one dataframe
stock = pd.concat(stock_dataframes, ignore_index=True)
stock.head()


,date,open,high,low,close,volume,ticker
0,2018-01-02,166.93,169.03,166.04,168.99,25555934,AAPL
1,2018-01-03,169.25,171.23,168.69,168.96,29517899,AAPL
2,2018-01-04,169.26,170.17,168.81,169.74,22434597,AAPL
3,2018-01-05,170.14,172.04,169.76,171.68,23660018,AAPL
4,2018-01-08,171.04,172.27,170.63,171.04,20567766,AAPL


In [3]:
# Clean and sort the combined data
stock['date'] = pd.to_datetime(stock['date'])
stock['volume'] = pd.to_numeric(stock['volume'], errors='coerce')
stock = stock.dropna(subset=['date', 'volume']).copy()
stock['volume'] = stock['volume'].astype(int)
stock = stock.sort_values(by=['date', 'ticker']).reset_index(drop=True)

stock.head()


,date,open,high,low,close,volume,ticker
0,2018-01-02,166.93,169.03,166.04,168.99,25555934,AAPL
1,2018-01-02,1172.00,1190.00,1170.51,1189.01,2694494,AMZN
2,2018-01-02,177.68,181.58,177.55,181.42,18151903,FB
3,2018-01-02,1048.34,1066.94,1045.23,1065.00,1237564,GOOG
4,2018-01-02,196.10,201.65,195.42,201.07,10966889,NFLX


In [4]:
# Save the cleaned combined data once
output_path = Path('/content/judith-6070/data/faanf.csv')
stock.to_csv(output_path, index=False)
print(f'Saved combined data to: {output_path}')


Saved combined data to: /content/judith-6070/data/faanf.csv


In [5]:
# Find the seven highest-volume trading days
top7 = stock.nlargest(7, 'volume')
top7[['date', 'ticker', 'volume']]


,date,ticker,volume
712,2018-07-26,FB,169803668
267,2018-03-20,FB,129851768
287,2018-03-26,FB,126116634
272,2018-03-21,FB,106598834
910,2018-09-21,AAPL,96246748
1225,2018-12-21,AAPL,95744384
1060,2018-11-02,AAPL,91328654


In [6]:
# Convert the data from wide to long format
stock_long = stock.melt(
    id_vars=['date', 'ticker'],
    var_name='variable',
    value_name='value'
)

stock_long.head(10)


,date,ticker,variable,value
0,2018-01-02,AAPL,open,166.93
1,2018-01-02,AMZN,open,1172.00
2,2018-01-02,FB,open,177.68
3,2018-01-02,GOOG,open,1048.34
4,2018-01-02,NFLX,open,196.10
5,2018-01-03,AAPL,open,169.25
6,2018-01-03,AMZN,open,1188.30
7,2018-01-03,FB,open,181.88
8,2018-01-03,GOOG,open,1064.31
9,2018-01-03,NFLX,open,202.05
